# YOLO26 Small Flag Detector Training Notebook

This notebook downloads the flag dataset from Google Drive, automatically configures paths for the Kaggle environment, and trains a **YOLO26 Small** flag detector model using Kaggle's GPU accelerators.

## Step 1: Install Dependencies
We install the `ultralytics` package (which provides YOLO26) and `gdown` (for downloading the Google Drive folder).

In [ ]:
!pip install -U ultralytics gdown PyYAML

## Step 2: Download Dataset from Google Drive
We download the Google Drive folder with ID `1eHAzVhdbyuChFYy0FnQxQYivN4EYkNQH` to `/kaggle/working/dataset`.

In [ ]:
import os
import shutil

# Create dataset directory
dataset_dir = "/kaggle/working/dataset"
os.makedirs(dataset_dir, exist_ok=True)

# Download the entire folder
!gdown --folder --id 1eHAzVhdbyuChFYy0FnQxQYivN4EYkNQH -O {dataset_dir}

## Step 3: Inspect and Configure Dataset YAML
We inspect the downloaded folder structure. If a `dataset.yaml` is present, we update its `path` property to point to the absolute directory. If no YAML config is found, we auto-generate one by scanning the dataset directories.

In [ ]:
import glob
import yaml

print("Scanning downloaded dataset structure...")
yaml_files = glob.glob(f"{dataset_dir}/**/*.yaml", recursive=True)
yaml_out_path = "/kaggle/working/dataset.yaml"

if yaml_files:
    yaml_path = yaml_files[0]
    print(f"Found existing YAML config at: {yaml_path}")
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    # Set absolute path to the directory containing the YAML config
    data['path'] = os.path.dirname(os.path.abspath(yaml_path))
    
    # Save the updated configuration to the root working directory
    with open(yaml_out_path, 'w') as f:
        yaml.safe_dump(data, f)
    print(f"Updated and saved YAML to: {yaml_out_path}")
else:
    print("No pre-existing YAML file found. Auto-detecting structure...")
    # Try to find 'images' folder
    images_dirs = glob.glob(f"{dataset_dir}/**/images", recursive=True)
    if images_dirs:
        dataset_root = os.path.dirname(os.path.abspath(images_dirs[0]))
        print(f"Detected dataset root containing 'images' folder: {dataset_root}")
        
        # Check all unique class IDs from text files to count classes
        txt_files = glob.glob(f"{dataset_root}/**/*.txt", recursive=True)
        class_ids = set()
        for txt in txt_files:
            if os.path.basename(txt) == 'classes.txt':
                continue
            try:
                with open(txt, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            class_ids.add(int(parts[0]))
            except:
                pass
                
        num_classes = max(class_ids) + 1 if class_ids else 1
        print(f"Detected {num_classes} classes.")
        
        yaml_data = {
            'path': dataset_root,
            'train': 'images',
            'val': 'images',
            'names': {i: f"class_{i}" for i in range(num_classes)}
        }
        
        # If splits exist
        if os.path.exists(os.path.join(dataset_root, 'images', 'train')):
            yaml_data['train'] = 'images/train'
            yaml_data['val'] = 'images/val' if os.path.exists(os.path.join(dataset_root, 'images', 'val')) else 'images/train'
            
        with open(yaml_out_path, 'w') as f:
            yaml.safe_dump(yaml_data, f)
        print(f"Generated new dataset config and saved to: {yaml_out_path}")
    else:
        print("Error: Could not locate 'images' directory. Please check downloaded folder structure.")

## Step 4: Verify GPU and Initialize YOLO26s Model
We load the **YOLO26 Small** model (`yolo26s.pt`) and check if CUDA is active.

In [ ]:
import torch
from ultralytics import YOLO

cuda_available = torch.cuda.is_available()
print(f"GPU (CUDA) Available: {cuda_available}")
if cuda_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    
model = YOLO("yolo26s.pt")

## Step 5: Start Model Training
We train the model on Kaggle's GPU. We default to **100 epochs** with a batch size of **16** and image size of **640**.

In [ ]:
device_val = 0 if torch.cuda.is_available() else 'cpu'

model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=device_val,
    workers=2,
    name="yolo26_small",
    exist_ok=True
)

## Step 6: Compress Outputs for Download
We package the final weights (`best.pt`, `last.pt`) and TensorBoard log charts into a single `.zip` file for easy download from Kaggle.

In [ ]:
import zipfile

results_dir = "/kaggle/working/runs/detect/yolo26_small"
zip_path = "/kaggle/working/yolo26_small_results.zip"

if os.path.exists(results_dir):
    print(f"Compressing results from {results_dir}...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(results_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # Add to zip with relative path
                zipf.write(file_path, os.path.relpath(file_path, results_dir))
    print(f"Saved package to: {zip_path}")
else:
    print("Error: Training runs directory not found.")